# 🍓🐧 Benchmark 03: Native CUDA C++ & cuBLAS (`03_cuda_cpp`) — UnB Metodologia Científica (Trabalho 2)

Bem-vindos ao terceiro e mais próximo do silício dos nossos experimentos no Google Colab Pro! ✨ Neste notebook, compilamos (`nvcc -O3 -lcublas`) e executamos a implementação **Nativa em C++/CUDA (`gpt_cuda.cu`)** (`implementations/03_cuda_cpp`) para o nosso **Decoder-Only Transformer (10.788.929 parâmetros)** no corpus *TinyShakespeare* (`vocab_size = 65`).

### 🔧 Arquitetura de Baixo Nível em CUDA C++ (`gpt_cuda.cu`)
- **Zero Overhead de Interpretador Python:** Loop de treinamento, forward, backward, passo do otimizador AdamW e geração autoregressiva rodam 100% em C++/CUDA nativo.
- **Gerenciamento Manual de VRAM (`cudaMalloc`):** Arena estática pré-alocada para parâmetros $\theta$, gradientes $\nabla\theta$, momentos do AdamW ($m, v$) e ativações de forward/backward.
- **GEMMs via `cublasSgemm` + Kernels `__global__` Customizados:** LayerNorm, Causal Multi-Head Attention, ReLU, Residual Add, Cross-Entropy e AdamW com redução em *shared memory*.

### 🎯 Invariantes Metodológicos
- **Semente Global Determinística (`SEED`):** `998244353` (carrega exatamente `golden_weights.bin` e `golden_batches.bin`!)
- **Precisão Numérica:** `FP32` IEEE 754 estrito com verificação de paridade $|L_0 - L_0^{\text{ref}}| < 10^{-4}$ e $|L_1 - L_1^{\text{ref}}| < 10^{-4}$.

## 🛠️ Célula 1: Verificação da GPU & Setup do Repositório GitHub 🍓

Checamos a GPU e o compilador CUDA no ambiente do Google Colab Pro e garantimos que o repositório esteja disponível localmente.

In [ ]:
!nvidia-smi

import os
from pathlib import Path

REPO_URL = "https://github.com/CalebMartim/gpt-implementations.git"
REPO_DIR = Path("trabalho_2")

if Path("common/generate_golden.py").exists() and Path("implementations/03_cuda_cpp/benchmark.py").exists():
    print("🍓 [Chuu Lab] Repositório já detectado no diretório atual:", Path.cwd())
elif (REPO_DIR / "common/generate_golden.py").exists():
    os.chdir(REPO_DIR)
    print("🍓 [Chuu Lab] Entrando no diretório existente do repositório:", Path.cwd())
else:
    print(f"🐧 [Chuu Lab] Clonando repositório de {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}
    if REPO_DIR.exists():
        os.chdir(REPO_DIR)
    print("🍓 [Chuu Lab] Diretório ativo:", Path.cwd())


## 🌟 Célula 2: Geração da Referência Dourada (`SEED = 998244353`) & Sanity Check 🐧

Geramos os arquivos binários `golden_weights.bin`, `golden_batches.bin` e `golden_meta.json` em `/tmp/golden_gpt` com a semente `998244353`.

In [ ]:
!python3 -m common.generate_golden --output-dir /tmp/golden_gpt --num-steps 5000 --eval-iters 50 --seed 998244353

import json
from pathlib import Path

golden_meta = json.loads(Path("/tmp/golden_gpt/golden_meta.json").read_text(encoding="utf-8"))
print("\n🍓 === Resumo do Golden Meta (SEED = 998244353) === 🐧")
print(json.dumps({
    "seed": golden_meta["seed"],
    "num_parameters": golden_meta["num_parameters"],
    "vocab_size": golden_meta["vocab_size"],
    "loss_step_0": golden_meta["loss_step_0"],
    "loss_step_1": golden_meta["loss_step_1"],
}, indent=2))


## 🚀 Célula 3: Compilação `nvcc` e Execução do Benchmark Completo (`03_cuda_cpp` — 5000 Steps + Inferência $B=1$ e $B=64$) 🍓

O script `implementations/03_cuda_cpp/benchmark.py` compila automaticamente `gpt_cuda.cu` com `nvcc -O3 -lcublas`, executa o binário nativo para 5000 steps de treinamento + inferência autoregressiva ($B=1$ e $B=64$) e anexa a contagem exata de SLOC em `results_03_cuda_cpp.json`.

In [ ]:
!python3 implementations/03_cuda_cpp/benchmark.py --golden-dir /tmp/golden_gpt --output-json results_03_cuda_cpp.json --max-iters 5000 --warmup-iters 50 --eval-interval 500 --eval-iters 50 --gen-tokens 1000


## 📊 Célula 4: Inspeção das Métricas e Download Automático do JSON 🐧

Exibimos os resultados de throughput, tempo wall-clock, consumo exato de VRAM do alocador nativo e SLOC, e fazemos o download automático de `results_03_cuda_cpp.json`.

In [ ]:
import json
from pathlib import Path

result_path = Path("results_03_cuda_cpp.json")
report = json.loads(result_path.read_text(encoding="utf-8"))

summary_view = {
    "implementation": report.get("implementation"),
    "gpu_name": report.get("gpu_name"),
    "sanity_check": report.get("sanity_check"),
    "train_time_s": report.get("train_time_s"),
    "train_tokens_per_sec": report.get("train_tokens_per_sec"),
    "infer_b1_tokens_per_sec": report.get("infer_b1_tokens_per_sec"),
    "infer_b1_latency_ms_per_token": report.get("infer_b1_latency_ms_per_token"),
    "infer_b64_tokens_per_sec": report.get("infer_b64_tokens_per_sec"),
    "peak_vram_mb": report.get("peak_vram_mb"),
    "final_train_loss": report.get("final_train_loss"),
    "final_val_loss": report.get("final_val_loss"),
    "sloc_core": report.get("sloc_core"),
    "sloc_harness": report.get("sloc_harness"),
    "sloc_total": report.get("sloc_total"),
}

print("🍓 === Resumo Final: 03_cuda_cpp === 🐧")
print(json.dumps(summary_view, indent=2))

try:
    from google.colab import files
    files.download("results_03_cuda_cpp.json")
except ImportError:
    print("🐧 Execução fora do Google Colab detectada; download automático ignorado.")
